# H3d — Retaliation

**H3d**  politicians who have been accused of lying are more likely to
subsequently accuse their accuser of lying in return.

### The question, and the one complication

Simple version: A accuses B. Does B accuse A back?

Count how often an accusation is followed by a reply from its target. The only
complication is knowing what to compare that number to. Pairs of MPs argue in
bursts — a flare-up produces accusations in *both* directions at once — so a high
"reply rate" on its own would just be measuring how often two people are
simultaneously angry.

The fix is a mirror: for every accusation, also look **backwards** by the same
number of days.

| | |
|---|---|
| reply rate **after** > rate **before** | retaliation: order matters |
| after ≈ before | just a burst: both directions cluster, no retaliation |

That is the whole design. Nothing else is needed.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys; sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lib import data, viz
viz.apply_style()

WINDOWS = [7, 30, 90, 365]      # days

## 1. The accusations

Every accusation aimed at a person, where we know **who accused whom**.

In [ ]:
con = data.duck()

ev = con.execute("""
    SELECT country, date,
           accuser_speaker_id AS a,
           target_speaker_id  AS b,
           COALESCE(is_interjection, 0) AS is_interjection
    FROM accusations
    WHERE target_type = 'person'
      AND accuser_speaker_id IS NOT NULL
      AND target_speaker_id  IS NOT NULL
      AND accuser_speaker_id <> target_speaker_id
      AND date IS NOT NULL AND length(date) >= 10
""").df()

ev["date"] = pd.to_datetime(ev["date"].str[:10], errors="coerce")
ev = ev.dropna(subset=["date"]).reset_index(drop=True)

print(f"accusations with both people identified : {len(ev):,}")
print(f"distinct accusers                       : {ev['a'].nunique():,}")
print(f"distinct targets                        : {ev['b'].nunique():,}")

## 2. Does the target hit back?

For each accusation A→B on day *t*, look for a B→A accusation:

- **after**: in the *t+1 … t+w* days
- **before**: in the *t−w … t−1* days  (the mirror image)
- **same day**: on day *t* itself, counted separately — that is one argument, not
  a considered reply

In [ ]:
# all B->A dates, keyed by the (A, B) accusation we are looking at
reverse = {k: np.sort(g["date"].values)
           for k, g in ev.groupby(["b", "a"])}


def reply_rates(events, windows=WINDOWS):
    out = []
    for w in windows:
        d = np.timedelta64(w, "D")
        after = before = same = 0
        for a, b, t in zip(events["a"], events["b"], events["date"].values):
            r = reverse.get((a, b))          # times B accused A
            if r is None:
                continue
            after  += bool(((r > t) & (r <= t + d)).any())
            before += bool(((r < t) & (r >= t - d)).any())
            same   += bool((r == t).any())
        n = len(events)
        out.append({"window_days": w,
                    "after_%": after / n * 100,
                    "before_%": before / n * 100,
                    "ratio": (after / before) if before else np.nan,
                    "same_day_%": same / n * 100})
    return pd.DataFrame(out)


rates = reply_rates(ev)
print(f"based on {len(ev):,} accusations\n")
print(rates.round(2).to_string(index=False))
print("\nafter_%  = share of accusations followed by a reply from the target")
print("before_% = same window, but looking backwards (the comparison)")
print("ratio    = after / before.  ~1.0 means no retaliation, just bursts.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(rates)); w = 0.38
ax.bar(x - w/2, rates["after_%"], w, label="reply AFTER", color="#c0603f")
ax.bar(x + w/2, rates["before_%"], w, label="mirror BEFORE", color="#9a9a9a")
ax.set_xticks(x)
ax.set_xticklabels([f"{d} days" for d in rates["window_days"]])
ax.set_ylabel("% of accusations")
ax.set_title("Is an accusation followed by a reply more often than preceded by one?")
ax.legend()
fig.tight_layout()
viz.savefig(fig, "h3d_reply_rates")
plt.show()

### The same comparison as a statistical test

The table above is a difference in proportions, but the two numbers are not
independent samples — they come from the *same* accusations, looked at forwards
and backwards. That makes it a **matched-pair** design, and the right test is
McNemar's.

Each accusation contributes one pair: (reply after?, reply before?). Pairs that
agree — a reply in both directions, or neither — carry no information about
order. Only the **discordant** pairs do:

- reply **after but not before** → evidence for retaliation
- reply **before but not after** → evidence against

Under the null that order does not matter, discordant pairs split 50/50. The
odds ratio is simply their ratio, and an exact binomial test gives the p-value.
SEs are clustered by dyad, since one pair of MPs can contribute many accusations.

In [ ]:
from scipy import stats
import statsmodels.api as sm


def matched_flags(events, w):
    """Per accusation: was there a reply after, and a reply before, within w days."""
    d = np.timedelta64(w, "D")
    aft, bef = [], []
    for a, b, t in zip(events["a"], events["b"], events["date"].values):
        r = reverse.get((a, b))
        if r is None:
            aft.append(0); bef.append(0); continue
        aft.append(int(((r > t) & (r <= t + d)).any()))
        bef.append(int(((r < t) & (r >= t - d)).any()))
    out = events[["a", "b", "country"]].copy()
    out["after"] = aft
    out["before"] = bef
    out["dyad"] = [f"{min(x, y)}|{max(x, y)}" for x, y in zip(out["a"], out["b"])]
    return out


rows = []
for w in WINDOWS:
    f = matched_flags(ev, w)
    n_af = int(((f["after"] == 1) & (f["before"] == 0)).sum())   # discordant, for
    n_bf = int(((f["after"] == 0) & (f["before"] == 1)).sum())   # discordant, against
    n_disc = n_af + n_bf
    orat = n_af / n_bf if n_bf else np.nan
    p = stats.binomtest(n_af, n_disc, 0.5).pvalue if n_disc else np.nan

    # same estimate as a model, with SEs clustered by dyad
    long = pd.concat([
        f.assign(y=f["after"],  is_after=1),
        f.assign(y=f["before"], is_after=0),
    ])
    long["obs"] = np.tile(np.arange(len(f)), 2)      # the matched pair id
    keep = long.groupby("obs")["y"].transform(lambda s: 0 < s.sum() < len(s))
    cl = long[keep]
    if cl["obs"].nunique() > 10:
        m = sm.ConditionalLogit(cl["y"], cl[["is_after"]], groups=cl["obs"]).fit(disp=False)
        lo, hi = m.conf_int().loc["is_after"]
        ci = f"[{np.exp(lo):.2f}, {np.exp(hi):.2f}]"
    else:
        ci = "-"

    rows.append({"window_days": w, "after_not_before": n_af,
                 "before_not_after": n_bf, "odds_ratio": orat,
                 "95% CI": ci, "p_exact": p})

mcn = pd.DataFrame(rows)
print(mcn.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("\nodds_ratio = discordant-for / discordant-against.")
print("  > 1 and p < .05  -> replies follow accusations more than they precede them")
print("  ~ 1              -> order carries no information; H3d not supported")

## 3. Robustness

**Interjections** are heckles: the accuser is the parsed heckler and the target is
assumed to be whoever held the floor. They happen inside a single debate, so they
inflate same-day exchanges in particular. Dropping them should not change the
after-vs-before comparison if that comparison is sound.

In [ ]:
no_int = ev[ev["is_interjection"] == 0]
reverse_all = reverse
reverse = {k: np.sort(g["date"].values)
           for k, g in no_int.groupby(["b", "a"])}
rates_ni = reply_rates(no_int)
reverse = reverse_all          # restore

print(f"excluding interjections ({len(no_int):,} accusations)\n")
print(rates_ni.round(2).to_string(index=False))

In [ ]:
# By country: is any single parliament driving the pattern?
rows = []
for c, g in ev.groupby("country"):
    if len(g) < 500:
        continue
    r = reply_rates(g, windows=[90]).iloc[0]
    rows.append({"country": c, "n": len(g),
                 "after_%": r["after_%"], "before_%": r["before_%"],
                 "ratio": r["ratio"]})
by_country = pd.DataFrame(rows).sort_values("ratio", ascending=False)
print("90-day window, countries with >= 500 accusations\n")
print(by_country.round(2).to_string(index=False))

## 4. Verdict

Read the **ratio** column in section 2.

- **ratio clearly above 1** — accusations are followed by replies more often than
  they are preceded by them. Order matters, so this is retaliation. **H3d supported.**
- **ratio ≈ 1** — replies are just as common before as after. The two directions
  cluster in time because pairs argue in bursts, and nothing here identifies
  retaliation. **H3d not supported** — though note this is a limit of what
  parliamentary timing data can show, not evidence that politicians never retaliate.

Report the `after_%` alongside it either way: the share of accusations that draw a
reply at all is a descriptive fact worth stating, whatever the ratio does.

Same-day exchanges are listed separately and should not be counted as
retaliation — they are single arguments, and for interjections partly an artefact
of how heckles are attributed.